## 📊 Part 3 — PCA for High-Dimensional Business Data: Retail Transactions

Real customer analytics often involves dozens of features: purchase frequency across categories,
recency scores, channel preferences, seasonal indices. PCA compresses these into a handful
of interpretable dimensions.

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import subprocess, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# Clone course repo to get bundled datasets
if not os.path.exists('pattern-recognition-notebooks'):
    print("Downloading course data...")
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', '--quiet',
         'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"Clone failed: {result.stderr[:200]}")
    else:
        print("✓ Course data ready")
else:
    print("✓ Course data already present")

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '../data'  # fallback for local use

print(f"  Data folder: {DATA_DIR}")
print(f"  Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {np.__version__}")


In [ ]:
# Synthetic retail customer feature matrix
# 500 customers × 20 features: category spend, frequency, recency, channel mix
np.random.seed(7)
n_customers = 500

# Underlying latent dimensions
price_sensitivity  = np.random.normal(0, 1, n_customers)  # PC1
engagement         = np.random.normal(0, 1, n_customers)  # PC2
channel_pref       = np.random.normal(0, 1, n_customers)  # PC3

features = {
    'grocery_spend':     2*engagement + price_sensitivity*0.5 + np.random.normal(0,.5,n_customers),
    'electronics_spend': engagement - price_sensitivity + np.random.normal(0,.6,n_customers),
    'clothing_spend':    engagement*0.8 + channel_pref*0.4 + np.random.normal(0,.5,n_customers),
    'home_spend':        engagement*0.6 + np.random.normal(0,.7,n_customers),
    'beauty_spend':      engagement*0.7 - channel_pref*0.3 + np.random.normal(0,.4,n_customers),
    'frequency':         engagement*1.2 + np.random.normal(0,.4,n_customers),
    'recency_days':      -engagement*0.8 + np.random.normal(0,.5,n_customers),
    'avg_basket':        engagement*0.5 - price_sensitivity*0.8 + np.random.normal(0,.5,n_customers),
    'returns_rate':      price_sensitivity*0.3 + np.random.normal(0,.4,n_customers),
    'promo_sensitivity': price_sensitivity*1.5 + np.random.normal(0,.3,n_customers),
    'mobile_pct':        channel_pref*1.2 + np.random.normal(0,.4,n_customers),
    'email_open_rate':   engagement*0.4 + channel_pref*0.5 + np.random.normal(0,.5,n_customers),
    'loyalty_points':    engagement*1.0 + np.random.normal(0,.4,n_customers),
    'weekend_pct':       channel_pref*0.3 + np.random.normal(0,.6,n_customers),
    'private_label_pct': price_sensitivity*0.8 + np.random.normal(0,.5,n_customers),
    'cross_category':    engagement*0.9 + np.random.normal(0,.4,n_customers),
    'seasonal_index':    np.random.normal(0,1,n_customers),
    'store_vs_online':   channel_pref*1.0 + np.random.normal(0,.4,n_customers),
    'referral_flag':     engagement*0.3 + np.random.normal(0,.8,n_customers),
    'complaint_count':  -engagement*0.4 + price_sensitivity*0.2 + np.random.normal(0,.5,n_customers),
}
X_retail = pd.DataFrame(features)
print(f"Retail customer feature matrix: {X_retail.shape}")
print(f"\n20 features per customer — hard to visualize or model directly")
print(f"PCA will find the key dimensions driving customer variation")

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardize (features are on different scales — spend vs rates vs counts)
scaler_r = StandardScaler()
X_scaled_r = scaler_r.fit_transform(X_retail)

# Fit PCA
pca_retail = PCA()
X_pca_r = pca_retail.fit_transform(X_scaled_r)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Scree plot
cumvar = np.cumsum(pca_retail.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1

axes[0].bar(range(1,21), pca_retail.explained_variance_ratio_*100,
            color='#1e5fa8', edgecolor='white', alpha=0.8, label='Individual')
axes[0].plot(range(1,21), cumvar*100, 'o-', color='#e85d20', lw=2,
             markersize=5, label='Cumulative')
axes[0].axhline(90, color='#888', lw=1, ls='--', alpha=0.6, label='90% threshold')
axes[0].axvline(n_90, color='#1a7a45', lw=1.5, ls='--',
                label=f'{n_90} PCs explain 90%')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('How Many Dimensions Capture Retail Customer Variation?')
axes[0].legend(fontsize=8)

# PC1 vs PC2 scatter — colored by engagement
axes[1].scatter(X_pca_r[:,0], X_pca_r[:,1],
                c=features['frequency'], cmap='RdYlGn', s=20, alpha=0.7)
axes[1].set_xlabel(f'PC1 ({pca_retail.explained_variance_ratio_[0]*100:.1f}% var)')
axes[1].set_ylabel(f'PC2 ({pca_retail.explained_variance_ratio_[1]*100:.1f}% var)')
axes[1].set_title('Customers in 2D PCA Space\n(color = purchase frequency)')
sm = plt.cm.ScalarMappable(cmap='RdYlGn')
plt.colorbar(sm, ax=axes[1], label='Purchase Frequency', shrink=0.8)

plt.tight_layout(); plt.show()
print(f"\n\u2714 {n_90} components capture 90% of customer variation (vs 20 original features)")
print(f"   Compression ratio: {20/n_90:.1f}x — huge simplification for downstream modeling")

In [ ]:
# What does each PC represent? — Loadings analysis
loadings = pd.DataFrame(pca_retail.components_[:3].T,
                        index=X_retail.columns,
                        columns=['PC1','PC2','PC3'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, pc in zip(axes, ['PC1','PC2','PC3']):
    sorted_load = loadings[pc].sort_values()
    colors = ['#e85d20' if v > 0 else '#1e5fa8' for v in sorted_load]
    ax.barh(sorted_load.index, sorted_load.values, color=colors, edgecolor='white')
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f'{pc} Loadings\n({pca_retail.explained_variance_ratio_[int(pc[-1])-1]*100:.1f}% variance)')
    ax.set_xlabel('Loading')

plt.suptitle('PCA Loadings: What Does Each Component Represent?\n'
             'Orange = positive contribution, Blue = negative',
             fontsize=11, y=1.02)
plt.tight_layout(); plt.show()
print("\n\u2714 PC1: high loadings on frequency, loyalty, cross-category = 'Engagement'")
print("   PC2: high loadings on promo sensitivity, private label = 'Price Sensitivity'")
print("   PC3: high loadings on mobile, store_vs_online = 'Channel Preference'")
print("   These match the latent dimensions we built into the data!")

In [ ]:
# @title 📝 Quiz — PCA
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What does the first principal component maximize?
# @markdown **Q2:** What do loadings tell you about a principal component?
# @markdown **Q3:** Why must features be standardized before PCA?
# @markdown **Q4:** You have 100 features. The first 5 PCs explain 85% of variance. What would you do next?
# @markdown **Q5:** What is one limitation of PCA for classification tasks?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

# Collect into answers dict for grading cell
answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the AI grading cell below for feedback")

### 📤 Submit your results

In [ ]:
_NB_NAME_="PCA"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")
